# Dendritic ANN nonlinearities on CIFAR-10 (Colab)

Runs control 1 (homogeneous type) and the per-dendrite ratio sweep on CIFAR-10 with a GPU.
The dataset is downloaded to this Colab machine only; nothing lands on your laptop except the
small results zip at the end.

**Before running:** Runtime > Change runtime type > GPU (a T4 is fine). Then run the cells top to bottom.
Free Colab sessions can time out; the sweep skips finished runs, so just re-run the cells. Mounting
Google Drive (optional cell below) keeps the results across sessions.

In [ ]:
!nvidia-smi -L
import torch
print("cuda available:", torch.cuda.is_available())

## 1. Get the code
Clones the `v1-nonlinearities` branch. Re-running this cell is safe.

In [ ]:
%cd /content
![ -d NEU231_Final_Project ] || git clone -b v1-nonlinearities https://github.com/HillaryKchao/NEU231_Final_Project.git
%cd /content/NEU231_Final_Project
!git pull -q
!pip install -q -r requirements.txt

## 2. Where to keep results
Set `USE_DRIVE = True` to store results in your Google Drive (survives a session timeout).
Otherwise they stay on this machine until you download the zip at the end.

In [ ]:
import os
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/neu231/results"
else:
    OUT = "/content/NEU231_Final_Project/results"
os.makedirs(OUT, exist_ok=True)
os.environ["OUT"] = OUT          # the shell cells below read $OUT
os.environ["DATASET"] = "cifar10"
print("results go to", OUT)

## 3. Smoke test (about a minute)
One epoch on 5,000 training images. Downloads CIFAR-10 to this machine's `data/` folder.
Expect `cuda` in the first line and a test accuracy around 30-35%.

In [ ]:
!python base_model.py --dataset cifar10 --dends 16 --soma 256 --epochs 1 --train-subset 5000 --out /tmp/smoke

## 4. Control 1 on CIFAR-10
LeakyReLU baseline, supra-only, sub-only, Ca-only and V1 mixed; seeds 1-3 at 16 x 256, 50 epochs each.
Roughly 2 hours on a T4. Already-finished runs are skipped, so re-run after a timeout.

In [ ]:
!DATASET=cifar10 OUT=$OUT bash sweep_controls.sh

## 5. Ratio sweep on CIFAR-10
One nonlinearity per dendrite at 12:3:1 (biological), 1:1:1 and 1:3:12; seeds 1-3. About an hour.

In [ ]:
!DATASET=cifar10 NONLINS=branch OUT=$OUT bash sweep_controls.sh

## 6. Figures and tables
Writes figures 1-8 and the CSVs for CIFAR-10 into `figures/cifar10/`, then inspects the V1 mixed model.

In [ ]:
!python plot_results.py --results $OUT --dataset cifar10 --out figures/cifar10
!python analyze_v1.py $OUT/dann_v1_local_cifar10_L1_d16_s256_seed1 --data-dir data --out figures/cifar10

In [ ]:
from IPython.display import Image, display
for f in ["7_control_homogeneous.png", "8_ratio_sweep.png"]:
    display(Image(f"figures/cifar10/{f}"))

## 7. Bring the results home
Zips the result JSONs (the numbers), the CIFAR-10 figures and the log, and downloads it.
The `.pt` weight files are left out (about 1 MB each); set `INCLUDE_WEIGHTS = True` if you want them.
Locally, unzip into the repo and run `python plot_results.py --results results --dataset cifar10 --out figures/cifar10`
to regenerate the figures from the JSONs.

In [ ]:
INCLUDE_WEIGHTS = False
pattern = "$OUT/*cifar10*" if INCLUDE_WEIGHTS else "$OUT/*cifar10*.json"
!rm -f /content/results_cifar10.zip
!zip -q -j /content/results_cifar10.zip {pattern} sweep_controls.log
!zip -q -r /content/results_cifar10.zip figures/cifar10
!ls -lh /content/results_cifar10.zip
from google.colab import files
files.download("/content/results_cifar10.zip")